In [1]:
import warnings
warnings.filterwarnings('ignore')

## [문제] 인터넷 신규 가입 여부 예측 하는 예제를 구현하시오.  

#### 제출할 파일명: 홍길동_0429_정확도.ipynb  

#### <처리 조건>
- logistic regression 모델을 사용한다.
- 데이터 로드 : pd.read_csv('newbie.csv') 
- 종속 변수 : 'Newbie'
- 독립 변수 제거 : 'who', 'Country', 'Years on Internet‘
- 결측값 처리 :
	- Age : 평균으로 채움
	- Household Income, Sexual Preference, Marital Status : 최빈값으로 채움
- 범주형 독립변수 원핫인코딩 처리 : pd.get_dummies() 사용
- test 데이터 사이즈는 20%, train 데이터는 80%로 한다.(train_test_split 사용할것)
- 최종적으로 모델을 만든 후 정확도(accuracy)를 출력하도록 한다.

#### 데이터 로드

In [2]:
import pandas as pd

newbie = pd.read_csv('newbie.csv')
newbie.head(50)

,who,Newbie,Age,Gender,Household Income,Sexual Preference,Country,Education Attainment,Major Occupation,Marital Status,Years on Internet
0,id74364,0,54.0,Male,$50-74,Gay male,Ontario,Some College,Computer,Other,4-6 yr
1,id84505,0,39.0,Female,Over $100,Heterosexual,Sweden,Professional,Other,Other,1-3 yr
2,id84509,1,49.0,Female,$40-49,Heterosexual,Washington,Some College,Management,Other,Under 6 mo
3,id87028,1,22.0,Female,$40-49,Heterosexual,Florida,Some College,Computer,Married,6-12 mo
4,id76087,0,20.0,Male,$30-39,Bisexual,New Jersey,Some College,Education,Single,1-3 yr
5,id77190,0,16.0,Male,NaN,Heterosexual,New York,Grammar,Education,Single,1-3 yr
6,id78923,0,67.0,Male,$30-39,Heterosexual,New Jersey,Other,Other,Married,1-3 yr
7,id78976,0,25.0,Male,Under $10,Heterosexual,Connecticut,College,Education,Single,1-3 yr
8,id89941,0,54.0,Male,$75-99,Heterosexual,Washington,Professional,Professional,Married,4-6 yr
9,id75957,1,48.0,Female,$50-74,Heterosexual,Ohio,College,Education,Married,Under 6 mo


#### 독립 변수 제거

In [3]:
newbie = newbie.drop(['who', 'Country', 'Years on Internet'], axis=1)
newbie.head(2)

,Newbie,Age,Gender,Household Income,Sexual Preference,Education Attainment,Major Occupation,Marital Status
0,0,54.0,Male,$50-74,Gay male,Some College,Computer,Other
1,0,39.0,Female,Over $100,Heterosexual,Professional,Other,Other


#### 결측값 처리

In [4]:
from sklearn.impute import SimpleImputer
import numpy as np

imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
age_mean = imp_mean.fit_transform(newbie[['Age']])

newbie['Age'] = age_mean
newbie.head(50)

,Newbie,Age,Gender,Household Income,Sexual Preference,Education Attainment,Major Occupation,Marital Status
0,0,54.000000,Male,$50-74,Gay male,Some College,Computer,Other
1,0,39.000000,Female,Over $100,Heterosexual,Professional,Other,Other
2,1,49.000000,Female,$40-49,Heterosexual,Some College,Management,Other
3,1,22.000000,Female,$40-49,Heterosexual,Some College,Computer,Married
4,0,20.000000,Male,$30-39,Bisexual,Some College,Education,Single
5,0,16.000000,Male,NaN,Heterosexual,Grammar,Education,Single
6,0,67.000000,Male,$30-39,Heterosexual,Other,Other,Married
7,0,25.000000,Male,Under $10,Heterosexual,College,Education,Single
8,0,54.000000,Male,$75-99,Heterosexual,Professional,Professional,Married
9,1,48.000000,Female,$50-74,Heterosexual,College,Education,Married


In [5]:
imp_most = SimpleImputer(missing_values=np.nan, strategy='most_frequent')

HI_most = imp_most.fit_transform(newbie[['Household Income']])
HI_most_newbie = pd.DataFrame(HI_most)
newbie['Household Income'] = HI_most_newbie[0]

SP_most = imp_most.fit_transform(newbie[['Sexual Preference']])
SP_most_newbie = pd.DataFrame(SP_most)
newbie['Sexual Preference'] = SP_most_newbie[0]

MS_most = imp_most.fit_transform(newbie[['Marital Status']])
MS_most_newbie = pd.DataFrame(MS_most)
newbie['Marital Status'] = MS_most_newbie[0]

newbie.head(50)

,Newbie,Age,Gender,Household Income,Sexual Preference,Education Attainment,Major Occupation,Marital Status
0,0,54.000000,Male,$50-74,Gay male,Some College,Computer,Other
1,0,39.000000,Female,Over $100,Heterosexual,Professional,Other,Other
2,1,49.000000,Female,$40-49,Heterosexual,Some College,Management,Other
3,1,22.000000,Female,$40-49,Heterosexual,Some College,Computer,Married
4,0,20.000000,Male,$30-39,Bisexual,Some College,Education,Single
5,0,16.000000,Male,$50-74,Heterosexual,Grammar,Education,Single
6,0,67.000000,Male,$30-39,Heterosexual,Other,Other,Married
7,0,25.000000,Male,Under $10,Heterosexual,College,Education,Single
8,0,54.000000,Male,$75-99,Heterosexual,Professional,Professional,Married
9,1,48.000000,Female,$50-74,Heterosexual,College,Education,Married


#### 범주형 독립변수 원핫인코딩 처리

In [6]:
newbie = pd.get_dummies(newbie, columns=['Gender', 'Household Income', 'Sexual Preference',
                               'Education Attainment', 'Major Occupation', 'Marital Status'],
                       dtype='int')
newbie.head()

,Newbie,Age,Gender_Female,Gender_Male,Household Income_$10-19,Household Income_$20-29,Household Income_$30-39,Household Income_$40-49,Household Income_$50-74,Household Income_$75-99,...,Major Occupation_Education,Major Occupation_Management,Major Occupation_Other,Major Occupation_Professional,Marital Status_Divorced,Marital Status_Married,Marital Status_Other,Marital Status_Separated,Marital Status_Single,Marital Status_Widowed
0,0,54.0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
1,0,39.0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
2,1,49.0,1,0,0,0,0,1,0,0,...,0,1,0,0,0,0,1,0,0,0
3,1,22.0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0,20.0,0,1,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,1,0


In [7]:
newbie.iloc[:, 1:]

,Age,Gender_Female,Gender_Male,Household Income_$10-19,Household Income_$20-29,Household Income_$30-39,Household Income_$40-49,Household Income_$50-74,Household Income_$75-99,Household Income_Over $100,...,Major Occupation_Education,Major Occupation_Management,Major Occupation_Other,Major Occupation_Professional,Marital Status_Divorced,Marital Status_Married,Marital Status_Other,Marital Status_Separated,Marital Status_Single,Marital Status_Widowed
0,54.0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
1,39.0,1,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,1,0,0,0
2,49.0,1,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,1,0,0,0
3,22.0,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,20.0,0,1,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19578,22.0,0,1,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,1,0
19579,19.0,0,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,1,0
19580,49.0,1,0,0,0,0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
19581,42.0,1,0,0,0,0,0,1,0,0,...,0,0,1,0,0,1,0,0,0,0


#### train 데이터, test 데이터 설정

In [118]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(newbie.iloc[:, 1:], newbie[['Newbie']],
                                                    test_size=0.2,
                                                    random_state=1000)

#### Logistic Regression (로지스틱 회귀)

In [119]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='sag')
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [120]:
score = model.score(X_test, y_test)
print(f'전체 데이터로 예측한 결과 : {score}')

전체 데이터로 예측한 결과 : 0.7638498851161604


In [121]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='liblinear')
model.fit(X_train_scaled, y_train)
score = model.score(X_test_scaled, y_test)
print(f'전체 데이터로 예측한 결과 : {score}')

전체 데이터로 예측한 결과 : 0.7641051825376564


In [122]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train_standard = scaler.transform(X_train)
X_test_standard = scaler.transform(X_test)

model = LogisticRegression(solver='liblinear')
model.fit(X_train_standard, y_train)
score = model.score(X_test_standard, y_test)
print(f'전체 데이터로 예측한 결과 : {score}')

전체 데이터로 예측한 결과 : 0.7633392902731683


#### for문 이용

In [128]:
from sklearn.model_selection import train_test_split

lst0 = []
for i in range(0, 1501):
    X_train, X_test, y_train, y_test = train_test_split(newbie.iloc[:, 1:], newbie[['Newbie']],
                                                        test_size=0.2,
                                                        random_state=i)
    model = LogisticRegression(solver='sag')
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    lst0.append(score)

print(max(lst0))

0.7806995149348992


In [131]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

lst1 = []
for i in range(0, 1501):
    X_train, X_test, y_train, y_test = train_test_split(newbie.iloc[:, 1:], newbie[['Newbie']],
                                                        test_size=0.2,
                                                        random_state=i)
    scaler = MinMaxScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(solver='liblinear')
    model.fit(X_train_scaled, y_train)
    score1 = model.score(X_test_scaled, y_test)
    lst1.append(score1)

print(max(lst1))    

0.7806995149348992


In [132]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

lst2 = []
for i in range(0, 1501):
    X_train, X_test, y_train, y_test = train_test_split(newbie.iloc[:, 1:], newbie[['Newbie']],
                                                        test_size=0.2,
                                                        random_state=i)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_standard = scaler.transform(X_train)
    X_test_standard = scaler.transform(X_test)
    
    model = LogisticRegression(solver='liblinear')
    model.fit(X_train_standard, y_train)
    score2 = model.score(X_test_standard, y_test)
    lst2.append(score2)

print(max(lst2))

0.7809548123563952


#### while문 사용

In [134]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

i = 1501
while i < 10001:
    X_train, X_test, y_train, y_test = train_test_split(newbie.iloc[:, 1:], newbie[['Newbie']],
                                                        test_size=0.2,
                                                        random_state=i)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_standard = scaler.transform(X_train)
    X_test_standard = scaler.transform(X_test)
    
    model = LogisticRegression(solver='liblinear')
    model.fit(X_train_standard, y_train)
    score3 = model.score(X_test_standard, y_test)
    if score3 > 0.7809:
        print('random_state :', i, 'accuracy :', score3)
    i += 1

random_state : 4280 accuracy : 0.7865713556293081
random_state : 4578 accuracy : 0.78580546336482
random_state : 9961 accuracy : 0.7817207046208833
